# Multimodal RAG- Beyond Text

**Module 8 · Lesson 8.9**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Describe → Embed - Turn an Image into Searchable Text
- Table Extraction - Structured Data into RAG
- Unified Multimodal Index - One Vector Space for Everything
- Direct Vision RAG - Send the Real Image at Query Time
- ColPali & ColQwen - Vision-First Document Retrieval
- Document AI - Unstructured, LlamaParse, DocTR, Tesseract

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install numpy byaldi "unstructured[pdf]" open_clip_torch torch faster-whisper google-cloud-vision google-genai python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_APPLICATION_CREDENTIALS", "") # 
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The Answer Was in the Chart

> 💡 **Info**
>
> A user asks your finance assistant: “What was Q4 revenue?” The number exists in the annual report - but only as a bar in a chart on page 12. The surrounding paragraph says “revenue grew steadily across the year” and never states the figure. Text RAG embeds and retrieves that paragraph, and the model answers “revenue grew steadily” - confidently useless, because **the actual number was never text**. It was pixels in a chart the pipeline never looked at. **Multimodal RAG fixes this: it has a vision model describe the chart (“Q4 = 135 Lakhs”), embeds that description alongside the text, and now the query finds it.** This lesson builds that pipeline from scratch, then tours the 2026 tools - ColPali, Document AI parsers, multimodal embeddings, and audio/video RAG - that make it production-grade.

### 🎯 What you will be able to do after this lesson

- **Build** multimodal RAG by hand - describe images to searchable text, extract tables to markdown, unify text + table + image in one vector space, and pass real images to Gemini at query time.

- **Compare** the two paradigms: describe-then-embed (text grounding) vs vision-first retrieval (ColPali/ColQwen and MaxSim), and know which document each wins.

- **Operate** the 2026 stack: Document AI parsers (Unstructured/LlamaParse), multimodal embeddings (CLIP/SigLIP 2/Nomic/BGE-VL), and audio/video pipelines (Whisper + frames + OCR).

- **Evaluate** production tradeoffs: cost/latency per modality, caching, modality-aware routing, and India document processing (Devanagari OCR, DigiLocker).

> 📦 **Info**
>
> 🧭 Before you start
> You need Lessons 8.1–8.3 (embeddings, chunking, vector retrieval - recalled below), a Google AI Studio key in `GOOGLE_API_KEY`, and `pip install google-genai pillow numpy` for the by-hand steps (plus `byaldi`, `unstructured`, `open_clip_torch`, and `faster-whisper` for the stack steps). Every block uses the current **google-genai** SDK: pre-2025 tutorials import the retired `google.generativeai` and call dead models, and error on today’s stack.

## 60-Second Warm-Up: What You Already Know

Three recalls, all load-bearing today. Click a card to check yourself.

> 👓 **Analogy**
>
> **Text-only RAG is reading a textbook with your eyes closed while someone reads the words aloud.** You hear every paragraph but miss every chart, diagram, and table - and the numbers that live only in those. Multimodal RAG opens your eyes: now you can see the revenue chart on page 12, the architecture diagram on page 34, and the pricing table on page 56. Same book - but now you see everything.
> **Where the analogy breaks down:** opening your eyes is instant and free; a machine “seeing” a chart means a vision-model call that costs money and 1–5 seconds. So multimodal RAG is not “look at everything” - it is *look at the visual parts once, at ingestion, turn them into text, and never pay for that look again at query time*. Deciding what deserves the expensive look is the whole engineering problem.

> 💡 **Info**
>
> ⚠️ Misconception: “multimodal RAG means embedding image pixels” (and “just send every page to a VLM”)
> Both wrong. The mature production pattern is **text grounding**: you have a vision model DESCRIBE each image/chart/table into text, then embed that text with your existing text embedder - images, tables, and paragraphs all land in one vector space, and your current vector store just works. Embedding raw pixels (CLIP, ColPali) is a distinct, heavier path you reach for only when layout itself carries meaning. And sending every page to a VLM is the cost trap: VLM description is 1–5s and the dominant expense, so you route text and tables to cheap extractors and reserve the VLM for genuinely visual content. Multimodal RAG is modality-aware routing, not “VLM everything.”

## Build One: Multimodal RAG by Hand

Steps 1–4 build multimodal RAG from scratch with nothing but google-genai: describe an image into searchable text, extract a table to markdown, unify text + table + image in one index, then pass the actual image to Gemini at query time. Building it once is what makes every later tool (ColPali, Unstructured, Nomic) legible - they are industrial versions of these four moves.

The Multimodal RAG pipeline (one extra stage over text RAG)

---

## 🎯 Concept 1: Describe → Embed - Turn an Image into Searchable Text

### Describe → Embed - Turn an Image into Searchable Text

A vision model describes the chart; the description becomes a searchable text chunk.

**Why this is step 1:** it is the single move that unlocks everything else. Retrieval can only rank things that live in its vector space, and that space is built from text. So you convert the image to text ONCE - a rich description - embed that, and now a chart is as findable as a paragraph. Master this and tables, audio, and video are the same trick with a different extractor.

> 🖹️ **Analogy**
>
> **You are writing the alt-text a librarian files under.** A blind patron can’t see the chart, so you write a precise caption: “bar chart, Netsetos revenue, Q1 45 to Q4 135 Lakhs, steady growth.” That caption goes in the card catalog; now anyone searching “Q4 revenue” finds the chart through your words.
> **Where the analogy breaks down:** a human caption-writer knows what matters; the vision model describes what you PROMPT it to, so a lazy prompt (“describe this image”) yields “a chart with some bars” and the exact numbers - the whole point - get lost. The description is only as searchable as the prompt is specific, which is why the code below asks for numbers, labels, and trends by name.

To make a revenue chart findable by the query “What was Q4 revenue?”, what do you store in the vector index?

- **describe_image_for_rag** sends the PIL image + a SPECIFIC prompt to Gemini (google-genai auto-converts the image) and returns a rich text description.

- **embed_content** embeds that DESCRIPTION (not the pixels) with the text embedder, so it lands in the same space as your text chunks.

**📝 `01_image_description.py Gemini`** - *Vision*

In [ ]:
# DESCRIBE AN IMAGE AS SEARCHABLE TEXT - vision to text embedding
# pip install google-genai pillow numpy
from google import genai
from PIL import Image, ImageDraw

client = genai.Client()   # reads GOOGLE_API_KEY from the environment

def describe_image_for_rag(image, context=""):
    prompt = ("Describe this image for a search index. State what kind of visual it is, "
              "every number, label, and trend, and what someone might search for.")
    if context:
        prompt += f"\nDocument context: {context}"
    # google-genai takes a PIL image directly in contents (auto-converted to a part)
    return client.models.generate_content(
        model="gemini-2.5-flash", contents=[prompt, image]).text.strip()

# a synthetic revenue bar chart
img = Image.new("RGB", (400, 250), "white"); d = ImageDraw.Draw(img)
d.text((20, 10), "Netsetos Revenue (Lakhs)", fill="black")
for i, (q, v) in enumerate([("Q1",45),("Q2",72),("Q3",98),("Q4",135)]):
    h = int(v * 1.2)
    d.rectangle([40+i*85, 220-h, 40+i*85+60, 220], fill="#0d9488")
    d.text((55+i*85, 225), q, fill="black"); d.text((50+i*85, 220-h-15), str(v), fill="black")

desc = describe_image_for_rag(img, context="Netsetos annual report 2024")
print("Description:", desc[:160], "...\n")

# embed the DESCRIPTION (not the pixels) with the text embedder
emb = client.models.embed_content(model="gemini-embedding-001", contents=desc).embeddings[0].values
print(f"Embedded description -> {len(emb)}-dim text vector; searchable by 'Q4 revenue', 'growth trend'.")

# Output:
# Description: A bar chart titled "Netsetos Revenue (Lakhs)" with four bars - Q1: 45, Q2: 72,
# Q3: 98, Q4: 135 - showing steady quarter-over-quarter growth ...
# Embedded description -> 3072-dim text vector; searchable by 'Q4 revenue', 'growth trend'.

#### 💡 What just happened

⚡ What just happened? Gemini Vision turned a bar chart into the sentence “Q1: 45, Q2: 72, Q3: 98, Q4: 135, steady growth,” and you embedded that TEXT as a vector. Now “What was Q4 revenue?” matches it by cosine similarity, and the answer the cold-open couldn’t find is retrievable. The tradeoff: the description is lossy - it captures what the prompt asked for and nothing else, so a question about a detail you didn’t prompt for still misses. Step 4 fixes precise-value questions by sending the actual image at generation time.

> 💡 **Info**
>
> ⛔ Anti-pattern: “embed the raw image pixels and answer straight from the vector”
> A tempting shortcut is to skip the description entirely, embed the chart’s pixels with a vision model, and expect the retrieved vector to somehow carry the answer. This is the **wrong way** for a text-first stack, and it **fails because** a pixel embedding lands in a SEPARATE image space (your text query can’t match it without a joint model), and even when it does retrieve, a similarity vector is not a readable number - “this looks like a revenue chart” is not “Q4 = 135.” Don’t do this: describe to text for retrieval, and pass the real image to the VLM only when you need to READ a value. Pixels retrieve by resemblance; the answer needs eyes on the original.

- Pick an image and watch the two paths: describe → text vector (works with your text store) vs embed raw pixels (needs a separate image index).
- Toggle a lazy vs specific prompt and see the description lose or keep the numbers.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Table Extraction - Structured Data into RAG

### Table Extraction - Structured Data into RAG

Extract tables to markdown, then embed as text like everything else.

**Why this is step 2:** tables are the modality text RAG mangles worst - naive extraction turns a clean grid into a soup of numbers with no rows or columns, and “which course is cheapest?” retrieves nothing useful. The fix mirrors step 1: convert the table to a STRUCTURED text form (markdown) that preserves rows and columns, then embed it. Same destination (text vector), different extractor.

> 📋 **Analogy**
>
> **A table is furniture; markdown is the flat-pack instructions.** You can’t email a bookshelf, but you can email the labeled parts-and-steps so it rebuilds perfectly. Markdown keeps the table’s structure (which price belongs to which course) in plain text the embedder and the LLM can both read.
> **Where the analogy breaks down:** flat-pack instructions are exact; a vision model reading a messy scanned table can merge cells, drop a row, or misread a digit - and unlike a chart, a wrong table number looks perfectly plausible. For high-stakes tables (financials), a structure-aware parser (step 6’s Unstructured/LlamaParse) beats a generic VLM prompt.

You extract a pricing table. Should you embed it as one markdown chunk, or split every cell into its own chunk?

- The synthetic table image is sent to Gemini with a prompt to extract it as markdown, ALL rows and columns.

- The returned markdown is embedded as a single chunk - rows and columns preserved - ready for queries like “cheapest course.”

**📝 `02_table_extraction.py Gemini`** - *Vision*

In [ ]:
# EXTRACT A TABLE TO MARKDOWN, THEN EMBED AS TEXT
# pip install google-genai pillow
from google import genai
from PIL import Image, ImageDraw

client = genai.Client()

# a synthetic table image
img = Image.new("RGB", (420, 180), "white"); d = ImageDraw.Draw(img)
for i, row in enumerate(["Course       | Price   | Hours | Rating",
                        "GenAI        | 14,999  | 146   | 4.8",
                        "Data Science | 12,999  | 120   | 4.7",
                        "Cloud GCP    | 11,999  | 110   | 4.6"]):
    d.text((15, 15+i*35), row, fill="black")

md = client.models.generate_content(model="gemini-2.5-flash",
    contents=["Extract this table as GitHub markdown. Include every row and column exactly.", img]).text.strip()
print(md, "\n")

# embed the whole markdown table as ONE chunk (keeps rows and columns together)
emb = client.models.embed_content(model="gemini-embedding-001", contents=md).embeddings[0].values
print(f"Embedded table as one {len(emb)}-dim chunk; searchable: 'cheapest course', 'highest rated'.")

# Output:
# | Course | Price | Hours | Rating |
# |---|---|---|---|
# | GenAI | 14,999 | 146 | 4.8 |
# | Data Science | 12,999 | 120 | 4.7 |
# | Cloud GCP | 11,999 | 110 | 4.6 |
# Embedded table as one 3072-dim chunk; searchable: 'cheapest course', 'highest rated'.

#### 💡 What just happened

⚡ What just happened? The grid became markdown that keeps every price next to its course, embedded as one chunk. Now “which course is cheapest?” retrieves the whole table and the LLM reads the structure to answer “Cloud GCP at 11,999.” The tradeoff vs a screenshot: markdown is portable and LLM-readable but a vision model can misread a scanned cell - so for financial tables you graduate to the structure-aware parsers in step 6, and always keep the table intact rather than splitting cells.

---

## 🎯 Concept 3: Unified Multimodal Index - One Vector Space for Everything

### Unified Multimodal Index - One Vector Space for Everything

Text chunks, image descriptions, and table markdown in one index. One retriever finds all.

**Why this is step 3:** steps 1 and 2 produced text from images and tables; now you drop all three - text, image-descriptions, table-markdown - into ONE index. The payoff is that the query doesn’t need to know where the answer lives. “Cheapest course?” (table), “students in 2025?” (chart), “refund policy?” (text) all go through the same retriever, because everything is a text vector now.

> 📚 **Analogy**
>
> **It is one card catalog for a mixed archive.** A library has books, maps, and photographs - wildly different objects - but a single card catalog describes each in words so one search finds any of them. Your unified index is that catalog: paragraphs, chart-captions, and table-markdown are all just cards in the same drawer.
> **Where the analogy breaks down:** a catalog card points you to the shelf, but the map itself still holds detail the card omits. Likewise your unified text index is great for FINDING the right chart, but to read an exact value off it you may still need to fetch the original image (step 4). The index locates; the original answers.

Text, image-descriptions, and table-markdown all become text vectors. How many retrievers does a query need?

- **add_text / add_image / add_table** each produce a text string (raw, described, or markdown) and store its embedding with a `type` tag.

- **query** embeds the question, ranks ALL entries by cosine similarity, and hands the top mix (text + table + image) to Gemini to answer.

- **query** embeds the question, scores every entry by dot product, and returns the top-k across all three modalities in one ranked list.

**📝 `03_multimodal_rag.py Full`** - *Pipeline*

In [ ]:
# UNIFIED MULTIMODAL RAG - text + tables + images in ONE index
# pip install google-genai pillow numpy
import numpy as np
from google import genai

client = genai.Client()

class MultimodalRAG:
    def __init__(self):
        self.entries = []   # each: {type, text, emb, source}

    def _embed(self, text):
        return np.array(client.models.embed_content(
            model="gemini-embedding-001", contents=text).embeddings[0].values)

    def add_text(self, text, source="doc"):
        self.entries.append({"type":"text", "text":text, "emb":self._embed(text), "source":source})

    def add_image(self, image, source="img", context=""):
        desc = client.models.generate_content(model="gemini-2.5-flash",
            contents=[f"Describe this image for search. Context: {context}", image]).text.strip()
        self.entries.append({"type":"image", "text":desc, "emb":self._embed(desc), "source":source})

    def add_table(self, markdown, source="table"):
        self.entries.append({"type":"table", "text":markdown, "emb":self._embed(markdown), "source":source})

    def query(self, question, k=3):
        qe = self._embed(question)
        order = sorted(range(len(self.entries)),
                       key=lambda i: -float(np.dot(qe, self.entries[i]["emb"])))
        top = [self.entries[i] for i in order[:k]]
        ctx = "\n\n".join(f"[{e['type'].upper()} {e['source']}]\n{e['text']}" for e in top)
        ans = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Answer from context only.\n\n{ctx}\n\nQ: {question}\nA:").text.strip()
        return {"answer": ans, "sources": [(e["type"], e["source"]) for e in top]}

rag = MultimodalRAG()
rag.add_text("Refund policy: full within 7 days, 50% for 8-30 days, none after 30.", "refund.txt")
rag.add_table("| Course | Price |\n|---|---|\n| GenAI | 14,999 |\n| Cloud GCP | 11,999 |", "pricing")
from PIL import Image, ImageDraw
chart = Image.new("RGB", (300,200), "white"); dd = ImageDraw.Draw(chart)
dd.text((10,5), "Students: 2024=2000, 2025=5000", fill="black")
rag.add_image(chart, "growth.png", context="annual report")

for q in ["Which course is cheapest?", "How many students in 2025?", "What is the refund policy?"]:
    r = rag.query(q)
    print(f"Q: {q}\nA: {r['answer'][:80]}  sources={r['sources']}\n")

# Output:
# Q: Which course is cheapest?
# A: Cloud GCP at 11,999.  sources=[('table', 'pricing'), ...]
# Q: How many students in 2025?
# A: 5,000 students.  sources=[('image', 'growth.png'), ...]

#### 💡 What just happened

⚡ What just happened? Three modalities, one index, one retriever. “Cheapest course?” pulled the table, “students in 2025?” pulled the image DESCRIPTION, “refund policy?” pulled the text - the query never knew the difference because all three are text vectors. The tradeoff: text grounding is simple and store-compatible, but every image was flattened to a description at ingestion, so precision questions about visual detail depend on how good that description was. When you need the exact pixel-level value, step 4 sends the real image.

- Fire a query and watch it light up the winning entry - text, table, or image - in a single shared vector space.
- See why the same retriever handles all three without knowing the modality.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Direct Vision RAG - Send the Real Image at Query Time

### Direct Vision RAG - Send the Real Image at Query Time

Retrieve with the description; generate with the actual image for exact values.

**Why this is step 4:** the description is lossy, so precise questions (“what’s the exact Q3 bar value?”) can fail even when retrieval found the right chart. The fix is a two-speed design: retrieve cheaply on text descriptions (Strategy A), then at generation time send the ACTUAL image to Gemini alongside the text (Strategy B) so it reads the real pixels. Best of both - cheap search, accurate answers.

> 🔍 **Analogy**
>
> **The card catalog finds the book; then you open the book.** You search the catalog (fast, text) to locate the right volume, but to quote an exact figure you pull the book off the shelf and read the page. Strategy A is the catalog; Strategy B is opening to the page - you only do it for the few results you actually need to read precisely.
> **Where the analogy breaks down:** opening a book is free; sending an image to the VLM at query time costs money and latency every single query. So you don’t pass images for every question - only when the query is about specific values in a chart or diagram. Routing that decision (does this query need the pixels?) is the real skill.

Retrieval found the right chart via its description. The user wants the exact Q3 value. What now?

- **add** stores a text description + its embedding + the original PIL image.

- **query** retrieves on the description (Strategy A), then appends the ACTUAL images to the prompt parts (Strategy B) so Gemini reads them.

- google-genai accepts a mixed list of strings and PIL images in `contents` - that is the whole multimodal prompt.

**📝 `04_vision_rag.py Gemini`** - *Vision*

In [ ]:
# DIRECT VISION RAG - retrieve on descriptions, generate with the real images
# pip install google-genai pillow numpy
import numpy as np
from google import genai

client = genai.Client()

class VisionRAG:
    def __init__(self):
        self.entries = []

    def _embed(self, text):
        return np.array(client.models.embed_content(
            model="gemini-embedding-001", contents=text).embeddings[0].values)

    def add(self, desc, source, image=None):
        self.entries.append({"desc":desc, "emb":self._embed(desc), "source":source, "image":image})

    def query(self, question, k=2):
        qe = self._embed(question)
        order = sorted(range(len(self.entries)),
                       key=lambda i: -float(np.dot(qe, self.entries[i]["emb"])))[:k]
        parts = [f"Answer from this context only.\nQuestion: {question}\nContext:"]
        for i in order:
            e = self.entries[i]
            parts.append(f"\n[{e['source']}] {e['desc']}")
            if e["image"] is not None:
                parts.append(e["image"])              # send the ACTUAL image (Strategy B)
        ans = client.models.generate_content(model="gemini-2.5-flash", contents=parts).text.strip()
        return {"answer": ans, "sources": [self.entries[i]["source"] for i in order]}

print("Strategy A (retrieve): image -> describe -> embed description -> text search (fast, cheap)")
print("Strategy B (generate): send the ACTUAL image to Gemini so it reads exact chart values")
print("Best practice: A for retrieval, B only when the query needs precise visual detail")

# Output:
# Strategy A (retrieve): image -> describe -> embed description -> text search (fast, cheap)
# Strategy B (generate): send the ACTUAL image to Gemini so it reads exact chart values
# Best practice: A for retrieval, B only when the query needs precise visual detail

#### 💡 What just happened

⚡ What just happened? Two strategies, one pipeline. Strategy A keeps retrieval cheap and text-only; Strategy B sends the real image at generation so Gemini can read a value the description never captured. The tradeoff is pure cost-vs-accuracy: A is fast and store-compatible; B is accurate but pays a vision call every query it runs on. The production answer is to route - A for everything, B only for queries that clearly need pixel-level precision. This describe-for-retrieval / look-for-generation split recurs through the rest of the lesson.

## The 2026 Multimodal Stack

You now own the concepts. Steps 5–10 map them onto the tools you ship with: ColPali’s vision-first retrieval, Document AI parsers, multimodal embeddings, audio/video pipelines, production hardening, and India-specific document processing. Same idea - industrial parts.

---

## 🎯 Concept 5: ColPali & ColQwen - Vision-First Document Retrieval

### ColPali & ColQwen - Vision-First Document Retrieval

Page screenshot → patch embeddings → MaxSim late interaction. No OCR pipeline.

**Why this is step 5:** everything so far flattened visuals to text at ingestion. ColPali (ICLR 2025) asks: what if you never OCR at all? It screenshots each page, encodes it into ~1,024 patch vectors with a vision model, and matches your query tokens directly against page patches. For dense, layout-heavy documents (financial filings, scientific PDFs, infographics) it captures what OCR+chunking throws away. The reference wrapper is **Byaldi** ([github.com/AnswerDotAI/byaldi](https://github.com/AnswerDotAI/byaldi)), used in the code below.

> 🖼️ **Analogy**
>
> **OCR retyping is transcribing a painting into words; ColPali is photographing it.** Transcription is lossy - layout, spatial relationships, and anything the transcriber didn’t think to note are gone. A photograph keeps the whole visual field, and ColPali’s MaxSim lets each query word point a flashlight at the exact patch of the page it matches.
> **Where the analogy breaks down:** a photo is one file; ColPali stores ~1,024 vectors PER PAGE, so a million pages is terabyte-scale - the storage bill OCR never had. And a photograph of an unfamiliar layout can confuse it where plain text would generalize. So ColPali is not a free upgrade: it wins on visual documents and loses on huge text-heavy corpora, which is why hybrid (ColPali finds the page, OCR text feeds the answer) is common.

ColPali stores ~1,024 patch vectors per page instead of one. What is the main cost of that?

- **from_pretrained** loads a ColQwen/ColPali checkpoint (validate on your own docs - some overfit their training set).

- **index** screenshots each PDF page and stores its patch embeddings - no OCR, no chunking.

- **search** runs MaxSim and returns page numbers + scores; you then pass the winning page IMAGES to a VLM to answer.

**📝 `05_colpali.py Byaldi +`** - *ColQwen*

In [ ]:
# COLPALI / COLQWEN - vision-first retrieval, no OCR pipeline
# pip install byaldi
from byaldi import RAGMultiModalModel

# load a ColPali-family checkpoint (ColQwen2 v1.0; ColPali-family checkpoints can overfit their training set - validate on YOUR docs)
rag = RAGMultiModalModel.from_pretrained("vidore/colqwen2-v1.0")

# index screenshots each page into ~1,024 patch vectors - no OCR, no chunking
rag.index(input_path="annual_report.pdf", index_name="report", overwrite=True)

# MaxSim late interaction: each query token matches its best page patch, scores summed
results = rag.search("What was Q4 revenue?", k=3)
for r in results:
    print(f"page {r.doc_id}:{r.page_num}  score {r.score:.2f}")

# then pass the winning PAGE IMAGE(s) to a VLM to actually read the answer
# (ColPali finds the page; a vision model reads it - the hybrid pattern)

# Output:
# page 0:12  score 18.42   <- the revenue chart page, found without any OCR
# page 0:3   score 12.07
# page 0:8   score 10.95

#### 💡 What just happened

⚡ What just happened? ColPali found the revenue-chart page with **zero OCR** - it screenshotted every page and matched query tokens to page patches via MaxSim (each query word finds its best-matching region, and the scores sum, so “Q4 revenue” lights up the exact chart patches). Note the scores aren’t 0–1 cosines - MaxSim sums per-token maxima. The tradeoff is storage: ~1,024 vectors/page is terabyte-scale at a million pages, so teams use small variants (ColSmol) or the hybrid pattern - ColPali retrieves the page, OCR text or a VLM generates the answer. Validate the checkpoint on your own documents; some overfit ViDoRe v1 (the reason the harder ViDoRe v2 exists).

- Step through query tokens and watch each one flashlight its best-matching page patch.
- Slide the patch count and compare MaxSim’s per-token matching against a single-vector cosine.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Document AI - Unstructured, LlamaParse, DocTR, Tesseract

### Document AI - Unstructured, LlamaParse, DocTR, Tesseract

Layout detection then per-element routing. Four tools, four niches.

**Why this is step 6:** your by-hand prompts work on clean synthetic images; real PDFs are messy - multi-column layouts, merged table cells, scanned skew. Document AI tools do the unglamorous heavy lifting: detect the layout (where are the tables, titles, figures?), then route each element to the right extractor. This is what turns a raw PDF into the clean text/markdown your steps 1–3 assume.

> 🏭 **Analogy**
>
> **A Document AI parser is a mailroom sorter.** Envelopes arrive in every shape; the sorter reads each one and routes it to the right desk - invoices to accounts, contracts to legal, junk to recycling. Unstructured’s layout model reads each PAGE ELEMENT and routes it: table to the table extractor, figure to the VLM, paragraph to OCR.
> **Where the analogy breaks down:** a mailroom sorter rarely mis-routes; layout detection does, especially on unusual designs, and a table sent to the text extractor comes out as scrambled numbers. So the tool choice is a real tradeoff: Tesseract is free but weak on layout, Unstructured hi_res is strong but heavier, LlamaParse leads on brutal tables but is hosted-only. Match the tool to the document, not the hype.

You must parse 50,000 scanned financial PDFs with brutal merged-cell tables, on-prem, no external API. Which tool leads?

| Tool | Best for | Key feature | Note |
|---|---|---|---|
| Unstructured.io | Production ETL | hi_res layout (YOLOX/Detectron2), element routing | Open + hosted |
| LlamaParse v2 | Brutal tables | Agentic parsing, multi-row headers, merged cells | Hosted-only (LlamaCloud) |
| DocTR | Custom OCR | PyTorch detection + recognition stages | Free (Apache 2.0) |
| Tesseract 5 | Simple / on-prem | 100+ languages, CPU-only | Free, weak on layout |

There is no universal best parser - benchmark on your own documents. **Unstructured** publishes open comparisons and the reference implementation at [github.com/Unstructured-IO/unstructured](https://github.com/Unstructured-IO/unstructured).

- **partition_pdf(strategy="hi_res")** runs layout detection to find titles, tables, figures, and text.

- **infer_table_structure=True** extracts tables as HTML you can embed instead of scrambled text.

- Each element carries a `category`, so you route tables, figures, and text to their optimal downstream handler.

**📝 `06_document_ai.py`** - *Unstructured*

In [ ]:
# DOCUMENT AI - layout-aware parsing that routes each element to the right extractor
# pip install "unstructured[pdf]"
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(
    filename="report.pdf",
    strategy="hi_res",                 # YOLOX/Detectron2 layout detection
    infer_table_structure=True,        # tables come out as HTML, not scrambled text
)

for el in elements[:4]:
    print(el.category, "|", str(el)[:55])

# route by category: Table -> embed el.metadata.text_as_html; NarrativeText -> chunk.
# (figures come as their own Image elements only with extract_image_block_types=["Image"])
tables = [el for el in elements if el.category == "Table"]
print(f"\nfound {len(tables)} table(s); HTML is in el.metadata.text_as_html")

# Output:
# Title | Netsetos Annual Report 2024
# NarrativeText | Revenue grew across all four quarters ...
# Table | Course Price Hours Rating GenAI 14,999 146 4.8 Data Science ...
# NarrativeText | The table above summarizes all four courses ...
# found 1 table(s); HTML is in el.metadata.text_as_html

#### 💡 What just happened

⚡ What just happened? Unstructured’s hi_res strategy ran layout detection, tagged each page element (Title, NarrativeText, Table), and preserved each table’s structure as HTML in `el.metadata.text_as_html` - the clean input steps 1–3 assumed (figures come as their own Image elements only when you pass `extract_image_block_types`). The tradeoff across tools is real: Tesseract is free but layout-blind, Unstructured hi_res is strong but heavier, LlamaParse leads on the worst tables but is hosted-only, DocTR gives you a custom PyTorch pipeline. There is no single best parser - you benchmark on YOUR documents and route by element category.

---

## 🎯 Concept 7: Multimodal Embeddings - CLIP, SigLIP 2, Nomic Vision, BGE-VL

### Multimodal Embeddings - CLIP, SigLIP 2, Nomic Vision, BGE-VL

Joint image-text spaces. Search images with text, without describing them first.

**Why this is step 7:** text grounding (steps 1–4) describes images to text before embedding. Multimodal embeddings skip the description: they put images AND text into one shared vector space directly, so a text query retrieves an image by similarity with no VLM caption in between. It is the other major paradigm, and for large image libraries it is faster and cheaper than captioning every picture.

> 🎭 **Analogy**
>
> **A joint embedding space is a bilingual dictionary where images and words share entries.** “Dog” and a photo of a dog map to nearly the same spot, so you can look up a word and find pictures, or look up a picture and find words - no translator in between. CLIP, SigLIP 2, and Nomic Vision are trained to build exactly this shared space.
> **Where the analogy breaks down:** a dictionary is precise; joint spaces are approximate - they capture “this looks like a revenue chart” but not “Q4 was 135.” So multimodal embeddings are great for FINDING relevant images fast, but you still hand the retrieved image to a VLM to read exact values. Retrieval and reading stay separate jobs.

You already have a text-embedding index. You want to add image search WITHOUT re-embedding all your text. What helps?

> ✅ **Info**
>
> 📈 The 2026 multimodal embedding landscape
> - **SigLIP 2 (Google):** dynamic-resolution (NaFlex), strong open image-text encoder; a good default over original CLIP.
> - **Nomic Embed Vision:** aligned to a frozen Nomic text encoder - **backward-compatible**, so you add image search without re-indexing existing text; Matryoshka dims (truncate the vector to a shorter length and it still works).
> - **BGE-VL (BAAI):** universal cross-modal (text→image, image→text, image+text→image).
> - **CLIP / OpenCLIP:** the original joint space; still a strong, easy baseline (used in the code below).

- **encode_image / encode_text** map a picture and a phrase into the SAME vector space.

- A text query and an image can be compared directly by cosine similarity - cross-modal retrieval with no caption step.

**📝 `07_multimodal_embeddings.py`** - *OpenCLIP*

In [ ]:
# MULTIMODAL EMBEDDINGS - image and text in ONE space (cross-modal search)
# pip install open_clip_torch pillow torch
import open_clip, torch
from PIL import Image

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k")
tokenizer = open_clip.get_tokenizer("ViT-B-32")

def embed_image(path):
    x = preprocess(Image.open(path)).unsqueeze(0)
    with torch.no_grad(): return model.encode_image(x)[0]

def embed_text(text):
    with torch.no_grad(): return model.encode_text(tokenizer([text]))[0]

# a TEXT query retrieves an IMAGE - both live in the same space
q = embed_text("a bar chart of quarterly revenue")
img = embed_image("chart.png")
sim = torch.cosine_similarity(q, img, dim=0).item()
print(f"text <-> image cosine similarity: {sim:.3f}")
print("High similarity => this image is retrievable by that text query, no caption needed.")

# Output:
# text <-> image cosine similarity: 0.31
# High similarity => this image is retrievable by that text query, no caption needed.

#### 💡 What just happened

⚡ What just happened? A text phrase and an image landed in the SAME vector space, so a text query retrieved an image directly - no VLM caption in between. That is the second paradigm: text grounding describes-then-embeds; multimodal embeddings embed both modalities natively. The tradeoff: joint spaces are cheap and fast for FINDING relevant images across a big library, but approximate - they can’t read exact values, so you still pass the retrieved image to a VLM to answer precisely. Nomic’s backward compatibility is the production sweetener: add image search without re-indexing your text.

- Drag a text query around a 2D embedding space and watch which images and words it pulls closest.
- See a picture of a chart and the phrase “revenue chart” sit near each other - cross-modal similarity made visible.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 8: Audio & Video RAG - Whisper, Frames, Timestamps

### Audio & Video RAG - Whisper, Frames, Timestamps

Transcribe to timestamped text; align ASR + key frames + on-screen OCR.

**Why this is step 8:** audio and video feel like a whole new problem, but they collapse to the same trick - text grounding - with one crucial addition: timestamps. You transcribe speech to timestamped text, describe key frames, OCR on-screen slides, align all three by time, and embed the text. Retrieval then returns not just an answer but the exact SECOND in the media to jump to.

> 🎬 **Analogy**
>
> **You are building a DVD scene-selection menu.** Nobody re-watches a two-hour film to find one line; the menu jumps you to the timestamp. Whisper writes the subtitles with times, key frames caption what’s on screen, OCR grabs the slide text - and every chunk carries the second it happened, so retrieval hands you the scene, not the whole reel.
> **Where the analogy breaks down:** DVD scenes are cut by a human editor at meaningful boundaries; your automatic chunker cuts by fixed windows or ASR pauses, which can split a sentence or merge two topics. So you chunk by scene/topic boundaries where you can, and always store timestamps so a slightly-off chunk still lands the user near the right moment.

You build RAG over lecture videos. Beyond the transcript text, what one piece of metadata is essential?

- **WhisperModel("large-v3-turbo")** loads the fast turbo model (int8 on CPU is fine for demos; use GPU/float16 or distil-large-v3 for long audio).

- **transcribe** returns segments that already carry `start`/`end` seconds (no `word_timestamps` needed unless you want per-word times).

- Embed each segment’s text and STORE its timestamp so retrieval can jump to the position.

**📝 `08_audio_video.py`** - *faster-whisper*

In [ ]:
# AUDIO RAG - transcribe to TIMESTAMPED chunks, then standard text RAG
# pip install faster-whisper
from faster_whisper import WhisperModel

model = WhisperModel("large-v3-turbo", device="cpu", compute_type="int8")
segments, info = model.transcribe("lecture.mp3")   # segments already carry .start/.end/.text

chunks = []
for seg in segments:
    chunks.append({"start": seg.start, "end": seg.end, "text": seg.text.strip()})
    print(f"[{seg.start:6.1f}s] {seg.text.strip()[:52]}")

# embed each chunk['text'] with your text embedder; KEEP the timestamp so
# retrieval returns the answer AND the second to jump to. Video adds two more
# streams (key-frame VLM descriptions + on-screen OCR) aligned by the same timestamp.

# Output:
# [   0.0s] Welcome to lesson eight point nine on multimodal RAG
# [   6.4s] Real documents are not just text - they have charts
# [  12.1s] Today we build a pipeline over images, tables, and audio

#### 💡 What just happened

⚡ What just happened? Whisper turbo transcribed the audio into timestamped segments; you embed the text and keep the timestamp, so “when did they explain MaxSim?” returns the answer AND “jump to 4:12.” Video is the same pattern with two more streams - key-frame descriptions and on-screen OCR - aligned by the shared timestamp. The tradeoff: three streams cost three extraction passes at ingestion, so you pre-process once and never re-transcribe at query time. The essential metadata is always the timestamp; without it you can answer but not point.

- Scrub a video timeline and watch the three streams - ASR transcript, key-frame captions, on-screen OCR - line up by time.
- Run a query and see retrieval jump the playhead to the matching second.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 9: Production - Cost, Latency, Caching, Routing

### Production - Cost, Latency, Caching, Routing

VLM description dominates latency and cost. Route, cache, pre-process.

**Why this is step 9:** the by-hand pipeline works on ten documents; production runs on ten million, where the VLM description that felt free is suddenly the dominant cost and the 1–5s latency spike. This step is the economics: route each modality to its cheapest sufficient extractor, cache aggressively, and pre-process visuals at ingestion so query time stays fast.

> 🚚 **Analogy**
>
> **Modality routing is a hospital triage desk.** Not every patient needs the MRI (the expensive VLM); the nurse routes a sprain to a quick X-ray and reserves the scanner for the cases that truly need it. Your router sends paragraphs to cheap OCR and reserves the vision model for the charts and figures that genuinely require it.
> **Where the analogy breaks down:** triage is a one-time decision per patient; your pipeline sees the SAME content repeatedly, so the bigger win is a cache - describe a chart once at ingestion, hash it, and never pay to describe it again. Routing decides WHO gets the expensive look; caching makes sure nobody gets it twice.

In a production multimodal pipeline, which stage dominates latency?

- **route** maps each modality to its cheapest sufficient extractor - the VLM is reserved for images.

- **EmbeddingCache** hashes content so repeated text/charts are served from cache, never re-embedded or re-described.

- Pre-process descriptions at ingestion; at query time you only embed the query and search.

**📝 `09_production.py Pure`** - *Python*

In [ ]:
# PRODUCTION - modality-aware routing + a content-hash embedding cache
import hashlib

def route(modality):
    # send each modality to its cheapest sufficient extractor; VLM only for images
    return {"text": "chunk+embed (ms, cheap)",
            "table": "extract->markdown (cheap)",
            "image": "VLM describe (1-5s, dominant cost)",
            "audio": "whisper transcribe",
            "video": "asr + frames + ocr"}[modality]

class EmbeddingCache:
    def __init__(self): self.store = {}
    def get_or_embed(self, text, embed_fn):
        key = hashlib.sha256(text.encode()).hexdigest()
        if key in self.store:
            return self.store[key], True          # cache hit - no API call
        self.store[key] = embed_fn(text)
        return self.store[key], False

print("router(image) ->", route("image"))
cache = EmbeddingCache()
hits = 0
for t in ["revenue chart", "revenue chart", "refund policy", "revenue chart"]:
    _, hit = cache.get_or_embed(t, lambda s: [len(s)])
    hits += int(hit)
print(f"cache hits: {hits}/4 (repeats served from cache, no re-embed)")

# Output:
# router(image) -> VLM describe (1-5s, dominant cost)
# cache hits: 2/4 (repeats served from cache, no re-embed)

#### 💡 What just happened

⚡ What just happened? The router sent images to the expensive VLM and everything else to cheap extractors, and the cache served the repeated “revenue chart” twice without re-embedding. That is the whole production game: **VLM description dominates cost and latency (1–5s)**, so you route to avoid it where you can, cache to avoid it twice, and pre-process at ingestion so query time is just embed-and-search. Content-hash caching commonly reclaims a large fraction of embedding calls; verify current model pricing before you size the bill.

- Slide the page count and the mix of text/table/image pages and watch cost, latency, and quality move for three pipelines.
- Compare OCR+text vs VLM-every-page vs ColPali - see why routing wins.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 10: India Multimodal RAG - Devanagari OCR, DigiLocker, Validation

### India Multimodal RAG - Devanagari OCR, DigiLocker, Validation

Bilingual scripts, ID cards, and the structured-data shortcut that skips OCR.

**Why this is step 10:** Indian document processing hits problems the English demos never surface - Devanagari is far harder to OCR than Latin script, ID cards mix Hindi and English with stamps and overlays, and formats vary across generations of cards. Getting this right, and knowing the DigiLocker shortcut that skips OCR entirely, is the difference between a demo and a product that ships in India.

> 🇮🇳 **Analogy**
>
> **OCR-ing an Aadhaar card is deciphering handwriting; DigiLocker is asking for the typed original.** Why squint at a photocopy when the issuing authority will hand you the clean, structured record on request? DigiLocker returns legally-valid XML for hundreds of millions of users’ documents - the same data with zero OCR error, when the document is available there.
> **Where the analogy breaks down:** the typed original isn’t always available - a random scanned invoice or an old card has no DigiLocker entry, so you fall back to OCR. And Devanagari OCR is genuinely hard (conjunct characters, matras, the connecting shirorekha line), so you pick a strong engine (Google Cloud Vision leads here) and then VALIDATE every extracted field with a regex before trusting it.

A user’s Aadhaar and PAN are already in DigiLocker. Do you OCR the uploaded card photo, or something else?

> 📦 **Info**
>
> 🌐 The India document stack
> - **Devanagari OCR is hard:** conjunct characters, matras above/below/beside the letter, and the shirorekha line connecting characters. General-purpose engines (Tesseract) struggle badly on handwritten Hindi; Google Cloud Vision is the reported leader, with PaddleOCR as a fallback (validate error rates on your own documents).
> - **DigiLocker shortcut:** hundreds of millions of users; legally-valid structured XML/PDF via API for Aadhaar, PAN, driving licence, academic records - skip OCR entirely when available.
> - **Bhashini / IndicPhotoOCR:** government-backed Indic language and scene-text tooling across many languages.
> - **Embeddings for Hindi RAG:** Qwen3-Embedding (reported #1 on the MTEB multilingual leaderboard) or BGE-m3.
> - **Always validate:** after OCR, check extracted fields with format rules (Aadhaar `\d{4}\s?\d{4}\s?\d{4}`, PAN `[A-Z]{5}[0-9]{4}[A-Z]`) before trusting them.

- **language_hints=["hi","en"]** tells Google Vision to expect bilingual Devanagari + English on the card.

- **regex validation** confirms the extracted Aadhaar/PAN match their known formats before you trust them.

- When the document exists in DigiLocker, skip all of this and pull the structured XML instead.

**📝 `10_india.py Cloud`** - *Vision*

In [ ]:
# INDIA MULTIMODAL RAG - bilingual OCR + structured-field validation
# pip install google-cloud-vision
import re
from google.cloud import vision

def ocr_hindi_english(path):
    client = vision.ImageAnnotatorClient()
    with open(path, "rb") as f:
        image = vision.Image(content=f.read())
    ctx = vision.ImageContext(language_hints=["hi", "en"])   # bilingual ID cards
    resp = client.document_text_detection(image=image, image_context=ctx)   # dense/document text
    return resp.full_text_annotation.text

# validate extracted ID fields BEFORE trusting them
AADHAAR = re.compile(r"\b\d{4}\s?\d{4}\s?\d{4}\b")
PAN = re.compile(r"\b[A-Z]{5}[0-9]{4}[A-Z]\b")

text = "Name: Asha   Aadhaar: 1234 5678 9012   PAN: ABCDE1234F"
print("aadhaar:", AADHAAR.search(text).group())
print("pan:    ", PAN.search(text).group())

# Best shortcut: DigiLocker API returns legally-valid structured XML for hundreds of millions of users -
# parse fields directly and skip OCR entirely when the document is available there.

# Output:
# aadhaar: 1234 5678 9012
# pan:     ABCDE1234F

#### 💡 What just happened

⚡ What just happened? Google Vision read the bilingual card with language hints, and a regex validated the Aadhaar and PAN formats before the fields were trusted - the guardrail that keeps a misread digit from silently corrupting your data. The bigger lesson: **when a document lives in DigiLocker, skip OCR entirely** and pull the structured XML - zero OCR error for hundreds of millions of users’ records. Devanagari OCR is genuinely hard (conjuncts, matras, shirorekha), so pick a strong engine, validate every field, and prefer structured data whenever it exists.

## Putting It Together

You turned text RAG multimodal by adding one stage - *extract and describe every modality into text* - then toured the tooling: ColPali’s vision-first retrieval, Document AI parsers, native multimodal embeddings, audio/video timestamp pipelines, production routing, and India document processing. The through-line: *a modality is only searchable once it lives in a vector space, so the whole craft is choosing how to get each one there - describe it to text (cheap, universal) or embed its pixels (heavier, layout-aware) - and looking at the original only when the answer needs the detail the description dropped.*

> 📦 **Info**
>
> 🔑 Key takeaways
> - **Describe, then embed.** The universal move: a vision model turns a chart/table/image into text, you embed the text, and one retriever searches every modality at once.
> - **Retrieve on descriptions, generate on originals.** Descriptions are lossy, so search cheaply on text (Strategy A) and pass the actual image at generation when a query needs exact values (Strategy B).
> - **Two paradigms.** Text grounding (describe→embed) vs pixel embeddings (CLIP/SigLIP/ColPali); ColPali wins on visually rich docs but costs multi-vector storage, so hybrid retrieve-then-read is common.
> - **Everything reduces to timestamped/structured text.** Audio and video become timestamped transcripts + frame descriptions + OCR; tables become markdown/HTML; the timestamp and the structure are the metadata that matter.
> - **Production is routing + caching.** VLM description dominates cost and latency, so route each modality to its cheapest sufficient extractor, cache by content hash, and pre-process at ingestion.
> - **India: prefer structured data.** Devanagari OCR is hard; use a strong engine, validate every field, and skip OCR entirely via DigiLocker when the document is available.

> ✅ **Info**
>
> 🗺️ Where this goes next
> - In Lesson 8.10 (**Agentic RAG**) an agent decides at runtime WHICH modality path to take - describe, look at the image, or run ColPali - instead of a fixed pipeline.
> - In Lesson 8.11 (**RAG Evaluation**) you measure multimodal retrieval quality - did the right chart or table get retrieved? - with a golden set and CI.
> - In Module 14 (**LLMOps**) the expensive ingestion pipeline (VLM description, transcription) becomes a monitored, scheduled production job with cost dashboards.

*Practice exercises are in the companion practice notebook.*

Eight exercises, easy to hard. Each builds or operates one layer of the multimodal-RAG stack.

---

## 🎓 Summary

You've completed the practical part of **Multimodal RAG- Beyond Text**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-8_9.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-8.9.html` - regenerate this notebook whenever the source HTML is updated.*
